In [ ]:
import torchvision.datasets as dsets
import torchvision.transforms as transforms

In [ ]:
train_dataset = dsets.MNIST(
    root="../data", train=True, transform=transforms.ToTensor(), download=True
)
test_dataset = dsets.MNIST(
    root="../data", train=False, transform=transforms.ToTensor(), download=True
)

In [ ]:
import torch.utils.data as Data

In [ ]:
num_epoch = 10
batch_size = 100
learning_rate = 0.005
num_test_img = 5

In [ ]:
train_loader = Data.DataLoader(
    dataset=train_dataset, batch_size=batch_size, shuffle=True
)

test_loader = Data.DataLoader(
    dataset=test_dataset, batch_size=batch_size, shuffle=False
)

In [ ]:
import torch.nn as nn

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self):
        super(AutoEncoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.Tanh(),
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, 12),
            nn.Tanh(),
            nn.Linear(12, 3),
        )
        self.decoder = nn.Sequential(
            nn.Linear(3, 12),
            nn.Tanh(),
            nn.Linear(12, 64),
            nn.Tanh(),
            nn.Linear(64, 128),
            nn.Tanh(),
            nn.Linear(128, 28 * 28),
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from torch.autograd import Variable

In [ ]:
autoencoder = AutoEncoder()
autoencoder

In [ ]:
import torch.optim as opt

In [ ]:
loss_fn = nn.MSELoss()
optimizer = opt.Adam(autoencoder.parameters(), lr=learning_rate)

In [ ]:
view_data = Variable(
    torch.Tensor(
        test_dataset.data[:num_test_img].view(-1, 28 * 28).type(torch.FloatTensor)
    )
    / 255.0
)

In [ ]:
for epoch in range(num_epoch):
    for i, (x, y) in enumerate(train_loader):
        b_x = Variable(x.view(-1, 28 * 28))
        b_y = Variable(x.view(-1, 28 * 28))
        b_labels = Variable(y)

        optimizer.zero_grad()
        encoded, decoded = autoencoder(b_x)
        loss = loss_fn(decoded, b_y)
        loss.backward()
        optimizer.step()

        if i % 100 == 0:
            print("epoch: %d, loss: %.4f" % (epoch, loss.data.item()))
            _, decoded_data = autoencoder(view_data)
            f, a = plt.subplots(2, num_test_img, figsize=(5, 2))
            for j in range(num_test_img):
                a[0][j].imshow(
                    np.reshape(view_data.data.numpy()[j], (28, 28)), cmap="gray"
                )
                a[0][j].set_xticks(())
                a[0][j].set_yticks(())
                a[1][j].imshow(
                    np.reshape(decoded_data.data.numpy()[j], (28, 28)), cmap="gray"
                )
                a[1][j].set_xticks(())
                a[1][j].set_yticks(())
            plt.draw()
            plt.show()

In [ ]:
m = nn.ConvTranspose2d(1, 1, kernel_size=2, stride=2)
a = torch.FloatTensor([[[[1, 2, 3], [4, 5, 6], [7, 8, 9]]]])
print(a.shape)

In [ ]:
b = m(a)
print(b.shape)
print(b)

In [ ]:
class AutoEncoderCNN(nn.Module):
    def __init__(self):
        super(AutoEncoderCNN, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=7, stride=1),
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=7, stride=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=2, stride=2),
            nn.Sigmoid(),
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

In [ ]:
autoencoderCnn = AutoEncoderCNN()
autoencoderCnn.cuda()

In [ ]:
loss_fn = nn.MSELoss()
optimizer = opt.Adam(autoencoderCnn.parameters(), lr=learning_rate)

In [ ]:
view_data = Variable(
    torch.Tensor(
        test_dataset.data[:num_test_img]
        .view(-1, 1, 28, 28)
        .type(torch.cuda.FloatTensor)
    )
    / 255.0
)

In [ ]:
for epoch in range(num_epoch):
    for i, (x, y) in enumerate(train_loader):
        b_x = Variable(x).cuda()
        b_y = Variable(x).cuda()
        b_labels = Variable(y)

        optimizer.zero_grad()
        encoded, decoded = autoencoderCnn(b_x)
        loss = loss_fn(decoded, b_y)
        loss.backward()
        optimizer.step()

        if i % 100 == 0:
            print("epoch: %d, loss: %.4f" % (epoch, loss.data.item()))
            _, decoded_data = autoencoderCnn(view_data)
            f, a = plt.subplots(2, num_test_img, figsize=(5, 2))
            for j in range(num_test_img):
                a[0][j].imshow(
                    np.reshape(view_data.data.cpu().numpy()[j], (28, 28)), cmap="gray"
                )
                a[0][j].set_xticks(())
                a[0][j].set_yticks(())
                a[1][j].imshow(
                    np.reshape(decoded_data.data.cpu().numpy()[j], (28, 28)),
                    cmap="gray",
                )
                a[1][j].set_xticks(())
                a[1][j].set_yticks(())
            plt.draw()
            plt.show()